In [2]:
# '취업'에 대해 학습 시키켜록한다
# 네이버 지식인에서 취업관 관련된 질문들을 가져와서 학습시키기 전 단계까지 처리하려고 한다

# 네이버 지식인에서 취업관 관련된 질문 200개를 추출해서 갸져오는 작업
import requests
from bs4 import BeautifulSoup
import pandas as pd

# 매개변수
# 변수명 : 타입 = 기본값
# def함수() -> 리턴타입:
def get_kin_simmary(keyword : str,page : int = 1)->pd.DataFrame:
	# URL 지정
	url = f"https://kin.naver.com/search/list.naver"
	# url뒤에 ?변수 = 값 형태로 만들기 위한 딕셔너리
	# 이 딕셔너리 이름은 반드시 parmas? X
	params = {
		'query' : keyword,
		'page' : page
	}
	# 왼쪽 params는 키워드 인자이가 때문반드시 params
	response = requests.get(url, params=params)

	try:
		# 상태 코드가 200이면 아무것도 없음. 200이 아니면 예외 발생
		response.raise_for_status()
		# 확인용
		# print(response.text)
		# Beautifulsoup 객체 생성

		# html 코드에서 원하는 요소를 탐색하기 위해 bs를 이용
		soup = BeautifulSoup(response.text, 'lxml')# lxml : html코드, lxml-xml

		#._searchListTitleAnchor들을 선택
		# select, semect_one : 선택자 기반으로 요소 선택(보통 html에서)
		# find_all, find : 태그 기반으로 요소 선택(보통 xml에서)
		link_elements = soup.select('._searchListTitleAnchor') # 최대 10개

		# 결과를 담을 리스트
		link_list, title_list = [], []

		# 반복문으로 선택한 요소를 활용
		for link_element in link_elements:
			# 확인용
			# print(link_element)
			title = link_element.text # 요소 안에 있는태그들은 무시한 순수 글자들
			# 요소.get('속성') : 속성 값을 가져옴
			link = link_element.get('href')

			# 확인용
			# print(title)
			# print(link)

			# 추출한 데이터를 각 리스트에 추가
			link_list.append(link)
			title_list.append(title)

			# 확인용
			# print(link_list)
			# df : 변수이름(dataFrame)
			# 딕셔너리 : {'변수명' : 값, '변수명' : 값}
			# 데이터 처리를 위한 데이터 프레임 생성
			df = pd.DataFrame({
				'고민 제목' : title_list,
				'link' : link_list
			})

			# 확인용
			# print(df)
		return df
	except Exception as e:
		print(f'예외발생 : {e}')
		return pd.DataFrame({
			'고민 제목' : [],
				'link' : []
		})
	

In [3]:
def get_content(url):
	headers = {
		'User-Agent' : 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36',
	}
	response = requests.get(url, headers=headers)
	try:
		response.raise_for_status()

	except Exception as e:
		print(f"예외 발생 {e}")
		return None

	soup = BeautifulSoup(response.text, 'lxml')
	content = soup.select_one('.questionDetail')

	return content.text

# print(get_kin_simmary(keyword, 1))
import pandas as pd
import time
max_page = 10
keyword = '고민'

# 모든 결과를 담을 데이터프레임
total_df = pd.DataFrame({
	'고민 제목' : [], 
	'link' : [],
	'고민 내용' : []
})
# 100개 가져올 때까지 반복 => 기본 10*10페이지
for page in range(1, max_page + 1):
	"""지식인 질문들을 질문 제목(title)과 내용(content)으로 구성된 데이터프레임으로 반환하는 """
	# 키워드, 페이지를 이용하여 검색된 결과에서 제목과 내용을 추출
	df = get_kin_simmary(keyword, page)

	contents = []
	# df에서 지직인 링크 목록을 가져와서 반복문으로 실행
	for link in df['link']:
		# { 'title' : "제목", 'content' : '질문 내용' }
		content = get_content(link) if link else None
		contents.append(content.strip())

	df['고민 내용'] = contents

	total_df = total_df.dropna()

	# 확인용
	# print(df)
	# 질문을 추출 => 질문이 어디에 있는지

	# 각 페이지 결과를 전체에 이어붙임
	total_df = pd.concat([total_df, df], axis=0)
	total_df = total_df.drop(columns=['link'])
	total_df.to_csv(f'{keyword}내용.csv', encoding="utf-8-sig", index=False)	

	# 지연
	time.sleep(0.5)

In [4]:
# index 리셋
total_df = total_df.reset_index(drop=True)
print(total_df)

                          고민 제목  \
0            틱장애 고민이네요.(대구 틱장애)   
1      성조숙증치료 고민되는데 꼭 해야 하는...    
2          백반증 치료방법 고민중(대구 백반증)   
3         여드름흉터 고민 해결(울산 여드름흉터)   
4        병원에서 진단 후 수술 고민하고 있습니다   
..                          ...   
95                     질염 증상 고민   
96                   청소년 정신과 고민   
97  코막힘수술 고민 중인데요.. 해야 할까요?...    
98       먹어야 하나 고민입니다.(노원구 관절염)   
99    안면비대칭 고민 리프팅으로 한쪽만 처진...    

                                                고민 내용  
0   대구 소아/남 틱장애아이 진료를 보고 왔는데 약을 써보자는 이야기를 들었는데요. 그...  
1   서울 10대 초반/여 성조숙증딸아이 검사 결과 성조숙증 초기라고 해서 성조숙증치료를...  
2   대구 30대 중반/여 백반증지금은 병원에서 레이저 치료를 꾸준히 받고 있는 중인데,...  
3   울산 50대 중반/남 여드름흉터여드름흉터가 많이 심한 편이라 이것 저것 치료 많이 ...  
4   역삼역 50대 초반/남 전립선비대증 치료요즘 소변을 봐도 시원하지 않고 밤마다 화장...  
..                                                ...  
95  지난 주말 남친과 처음으로 관계를 가지게 되었는데  그날 이후로 질염 증상이 나타나...  
96  고3 학생입니다방학때부터 그러더니 개학 후 맛이 갔습니다.정확한 상태를 말씀드리면....  
97  용인 30대 초반/여 코막힘코막힘이 너무 심해서 요즘 진짜 숨 쉬는 게 힘들어요. ...  
98  노원구 50대 초반/남 관절

In [7]:
# 추출한 데이터 전처리 작업
# 중복 데이터 제거
clean_df = total_df.drop_duplicates(subset=['고민 제목'])

# 결측치 처리
clean_df = clean_df.dropna()

mean_len = 5

final_df = clean_df.loc[clean_df['고민 제목'].str.len() > mean_len]

# 인덱스 리셋
final_df = final_df.reset_index(drop=True)
print(final_df)

                          고민 제목  \
0            틱장애 고민이네요.(대구 틱장애)   
1      성조숙증치료 고민되는데 꼭 해야 하는...    
2          백반증 치료방법 고민중(대구 백반증)   
3         여드름흉터 고민 해결(울산 여드름흉터)   
4        병원에서 진단 후 수술 고민하고 있습니다   
..                          ...   
95                     질염 증상 고민   
96                   청소년 정신과 고민   
97  코막힘수술 고민 중인데요.. 해야 할까요?...    
98       먹어야 하나 고민입니다.(노원구 관절염)   
99    안면비대칭 고민 리프팅으로 한쪽만 처진...    

                                                고민 내용  
0   대구 소아/남 틱장애아이 진료를 보고 왔는데 약을 써보자는 이야기를 들었는데요. 그...  
1   서울 10대 초반/여 성조숙증딸아이 검사 결과 성조숙증 초기라고 해서 성조숙증치료를...  
2   대구 30대 중반/여 백반증지금은 병원에서 레이저 치료를 꾸준히 받고 있는 중인데,...  
3   울산 50대 중반/남 여드름흉터여드름흉터가 많이 심한 편이라 이것 저것 치료 많이 ...  
4   역삼역 50대 초반/남 전립선비대증 치료요즘 소변을 봐도 시원하지 않고 밤마다 화장...  
..                                                ...  
95  지난 주말 남친과 처음으로 관계를 가지게 되었는데  그날 이후로 질염 증상이 나타나...  
96  고3 학생입니다방학때부터 그러더니 개학 후 맛이 갔습니다.정확한 상태를 말씀드리면....  
97  용인 30대 초반/여 코막힘코막힘이 너무 심해서 요즘 진짜 숨 쉬는 게 힘들어요. ...  
98  노원구 50대 초반/남 관절